# 3. Genomics and transcriptomics on real data (cBioPortal)

TCGA uveal melanoma (PanCancer Atlas) via the cBioPortal REST API: driver mutation frequency and driver mRNA z-scores split by BAP1 (the metastatic-risk axis). Real data, no simulation.

In [ ]:
import requests, pandas as pd, numpy as np, matplotlib.pyplot as plt
from collections import defaultdict
from sklearn.decomposition import PCA
API = "https://www.cbioportal.org/api"; STUDY = "uvm_tcga_pan_can_atlas_2018"; SL = STUDY + "_all"
PANEL = {2776: "GNAQ", 2767: "GNA11", 8314: "BAP1", 23451: "SF3B1", 1964: "EIF1AX", 57105: "CYSLTR2", 5332: "PLCB4"}
N = len(requests.get(f"{API}/studies/{STUDY}/samples", timeout=40).json()); print("tumors:", N)

## Genomics: driver mutation frequency

In [ ]:
muts = requests.post(f"{API}/molecular-profiles/{STUDY}_mutations/mutations/fetch",
   json={"sampleListId": SL, "entrezGeneIds": list(PANEL)}, params={"projection": "SUMMARY"}, timeout=60).json()
ms = defaultdict(set)
for m in muts: ms[PANEL[m["entrezGeneId"]]].add(m["sampleId"])
freq = pd.Series({g: round(100*len(ms[g])/N, 1) for g in PANEL.values()}).sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 4.5)); ax.bar(freq.index, freq.values, color="#2f5fe0")
ax.set_ylabel("% mutated"); ax.set_title(f"Uveal melanoma drivers (TCGA, n={N})"); plt.xticks(rotation=40, ha="right"); plt.tight_layout()
print("GNAQ or GNA11:", round(100*len(ms["GNAQ"] | ms["GNA11"])/N, 1), "% ; overlap:", len(ms["GNAQ"] & ms["GNA11"]))
freq

## Transcriptomics: driver mRNA z-scores and a BAP1 split

In [ ]:
rp = next(p["molecularProfileId"] for p in requests.get(f"{API}/studies/{STUDY}/molecular-profiles", timeout=40).json()
          if "rna_seq" in p["molecularProfileId"] and "median_Zscores" in p["molecularProfileId"] and "normal" not in p["molecularProfileId"])
rna = requests.post(f"{API}/molecular-profiles/{rp}/molecular-data/fetch",
   json={"sampleListId": SL, "entrezGeneIds": list(PANEL)}, timeout=60).json()
df = pd.DataFrame([{"sample": r["sampleId"], "gene": PANEL[r["entrezGeneId"]], "z": r["value"]} for r in rna])
mat = df.pivot_table(index="gene", columns="sample", values="z").reindex(list(PANEL.values())).dropna(how="all")
bap1 = mat.loc["BAP1"]; status = np.where(bap1 < bap1.median(), "BAP1-low", "BAP1-high")
p = PCA(2).fit_transform(np.nan_to_num(mat.T.values))
fig, ax = plt.subplots(figsize=(6, 5))
for lab, c in [("BAP1-low", "#c0392b"), ("BAP1-high", "#2f5fe0")]:
    mask = status == lab; ax.scatter(p[mask, 0], p[mask, 1], c=c, s=45, edgecolor="w", label=lab)
ax.legend(frameon=False); ax.set_xlabel("PC1"); ax.set_ylabel("PC2"); ax.set_title("Uveal melanoma transcriptomes by BAP1"); plt.tight_layout()

BAP1-low tumors correspond to the class 2, high-metastatic-risk molecular subtype of uveal melanoma.

---
*Disclaimer: this notebook produces research and educational analysis of public data, not medical advice. Confidence and validation scores summarize evidence strength, not the probability of benefit for any individual. Every clinical decision must be made by a qualified health care provider, ideally within a clinical trial.*

*Developed by Dig Vijay Kumar Yarlagadda, [digvijayky.com](https://digvijayky.com).*